# harvest

> turn a page's own JSON API into vault data

In [ ]:
#| default_exp harvest

In [ ]:
#| hide
from nbdev.showdoc import *

Listing and product pages render from an internal JSON API. Reading that API paginates cleanly and
survives redesigns; scraping the DOM does neither. Each record becomes its own `##` section, so it
is retrievable on its own.

In [ ]:
#| export
import json, re, time
from urllib.parse import urlparse
from fastcore.all import AttrDict, L, Path, first, patch
from vishalakshi.core import Vault

In [ ]:
#| export
TITLE_KEYS = ('name', 'title', 'displayName', 'productName', 'description', 'label',
              'heading', 'headline', 'sku', 'id')

def pick_title(rec:dict, keys=TITLE_KEYS) -> str:
    'The most title-like field in a record, else a short digest of it.'
    for k in keys:
        v = rec.get(k)
        if isinstance(v, str) and v.strip(): return re.sub(r'\s+', ' ', v.strip())[:120]
    return re.sub(r'\s+', ' ', json.dumps(rec, default=str))[:80]

def flatten_record(rec, prefix='', max_depth:int=3, _d:int=0) -> list:
    'A record as flat `(dotted.key, value)` pairs — nested JSON made searchable as text.'
    out = []
    if _d >= max_depth or not isinstance(rec, (dict, list)):
        return [(prefix.rstrip('.'), rec)] if prefix else []
    items = rec.items() if isinstance(rec, dict) else enumerate(rec[:12])
    for k, v in items:
        p = f'{prefix}{k}.'
        if isinstance(v, (dict, list)) and v: out += flatten_record(v, p, max_depth, _d+1)
        elif v not in (None, '', [], {}): out.append((p.rstrip('.'), v))
    return out

def records_md(records, title_keys=TITLE_KEYS, max_fields:int=40) -> str:
    """Render JSON records as markdown, one `##` section per record.

    The heading matters more than it looks: `build_tree` turns each `##` into a tree node, so every
    record becomes its own retrievable section with its own breadcrumb, rather than one giant blob
    that only ever matches as a whole."""
    parts = []
    for rec in L(records):
        if not isinstance(rec, dict): rec = {'value': rec}
        fields = flatten_record(rec)[:max_fields]
        body = '\n'.join(f'- {k}: {v}' for k, v in fields)
        parts.append(f'## {pick_title(rec, title_keys)}\n\n{body}')
    return '\n\n'.join(parts)

def find_records(data, min_len:int=2):
    """The list of records inside an arbitrary JSON response.

    APIs bury their payload at different depths (`results`, `data.products.items`, a bare array),
    so this walks for the longest list of dicts rather than guessing a key name."""
    best = []
    def walk(o, d=0):
        nonlocal best
        if d > 6: return
        if isinstance(o, list):
            ds = [x for x in o if isinstance(x, dict)]
            if len(ds) >= min_len and len(ds) > len(best): best = ds
            for x in o[:20]: walk(x, d+1)
        elif isinstance(o, dict):
            for v in o.values(): walk(v, d+1)
    walk(data)
    return best

In [ ]:
#| export
@patch
def apis(self:Vault,
         url:str,             # page to watch
         pattern:str='*',     # glob/regex filtering captured request URLs
         session:bool=False,  # capture through the logged-in debug Chrome instead of a throwaway browser
         preview:int=240,     # chars of each response shown
         **kw                 # forwarded to fossick.find_xhr
) -> L:
    """Discover the JSON endpoints a page calls, so you can read its data instead of its HTML.

    Most product, listing and dashboard pages render from an internal JSON API. Reading that API is
    faster, paginates cleanly and survives redesigns, where scraping the DOM does none of those.
    Captures are kept on the vault so `harvest(capture=i)` can replay one."""
    from fossick.core import find_xhr
    hits = L(find_xhr(url, pattern=pattern, session=session, **kw))
    self._captures = [h.get('capture') for h in hits]
    return L(AttrDict(n=i, url=h['url'], content_type=h.get('content_type'),
                      records=len(find_records(h.get('data'))),
                      preview=json.dumps(h.get('data'), default=str)[:preview])
             for i, h in enumerate(hits))

In [ ]:
#| export
@patch
def harvest(self:Vault,
            url:str,                       # the page whose API you want
            pattern:str='*',               # which captured request URLs to keep
            title:str=None,                # document title; defaults to the page host + path
            capture:int=None,              # replay a specific endpoint from the last apis() call
            pages:int=1,                   # pages to pull; >1 paginates the endpoint
            page_field:str='page',         # query/body key incremented per page
            method:str=None,               # override the captured method
            session:bool=False,            # capture through the logged-in Chrome
            title_keys=TITLE_KEYS,         # fields to use as each record's heading
            force:bool=False,
            **kw
) -> dict:
    """Sniff a page's JSON API, pull the records, and file them in the vault as `kind='data'`.

    One document per harvest, one section per record, so a catalogue becomes individually
    retrievable rows that sit alongside your notes and papers in the same searches. Re-harvesting
    the same source replaces rather than duplicates when `force=True`; otherwise it is a no-op,
    which is what makes this safe to put behind a `watch`."""
    from fossick.core import replay_xhr, paginate_api
    caps = getattr(self, '_captures', None)
    if capture is None or not caps:
        found = self.apis(url, pattern=pattern, session=session)
        if not found: return dict(url=url, skipped='no JSON endpoints captured')
        capture = max(range(len(found)), key=lambda i: found[i].records)
        caps = self._captures
    cap = caps[capture]
    if cap is None: return dict(url=url, skipped=f'capture {capture} is not replayable')
    ep = cap.get('url')
    if pages > 1:
        items = paginate_api(ep, method=method or (cap.get('method') or 'GET').upper(),
                             page_field=page_field, max_pages=pages, **kw)
    else:
        r = replay_xhr(cap, **kw)
        try: data = r.json()
        except Exception: data = None
        items = find_records(data)
    if not items: return dict(url=url, endpoint=ep, skipped='no records found in the response')
    ttl = title or f'{urlparse(url).netloc}{urlparse(url).path}'.strip('/')
    md = f'# {ttl}\n\n{records_md(items, title_keys)}'
    return dict(self.add(md, ttl, source=ep, kind='data', force=force,
                         meta=dict(page=url, endpoint=ep, records=len(items),
                                   harvested_at=time.time())),
                endpoint=ep, records=len(items))

@patch
def add_records(self:Vault, records, title:str, source:str=None, kind:str='data',
                title_keys=TITLE_KEYS, force:bool=False, meta:dict=None) -> dict:
    'File a list of dicts you already have (any API, any export) as one document, one section each.'
    md = f'# {title}\n\n{records_md(records, title_keys)}'
    return self.add(md, title, source=source or f'records:{title}', kind=kind, force=force,
                    meta=dict(meta or {}, records=len(L(records))))